# WASP2026 draft: how many questions, and how many came back unparsable

Three counts, all read off disk rather than retyped, so none of them can drift
from the released dataset or the runs:

1. **the whole benchmark** -- every question in every released qa json
   (`VQA_full/qa_jsons/`), i.e. what the dataset contains whether or not any
   model was ever shown it
2. **what was actually asked** -- the questions in the n150 run pickles, for
   the `original` (clean figures) and `archive-light` (aged figures) runs
3. **unparsable answers** -- for each of the three models in each of those two
   runs, how many responses no answer could be pulled out of

On (3): "unparsable" here means `answer_for()` returns `None` -- the response
is not a wrong answer, it is not an answer at all.  That is the union of
`parse_llm_dual_json` failing outright (`PARSE_ERROR`) and it succeeding but
yielding nothing usable (not a dict, empty, or explanation-only).  A model that
answers the right thing under a renamed key is **not** counted here:
`answer_for` matches the requested key up to spacing/underscore/case, so
Claude's `plot_types` for `plot types` resolves normally.  See
`models/utils/parse_lmm_output_utils.py` for why that is the right line to
draw.


In [1]:
# ======================== CONFIG ========================
# The released dataset: one json per figure, holding both the figure's own
# parameters and the full question set generated from them.
qa_jsons_dir = '~/astro_sky_image_vqa/VQA_full/qa_jsons/'

# The runs, as (label, directory of per-model pickles).  Both cover the SAME
# 150 figures -- they differ only in whether the page was aged before it was
# sent -- so a difference between them is a difference in the image, not in the
# question set.  'archive' (full-strength aging) is deliberately left out; add
# it here if it ends up in the paper.
runs = [
    ('original',      '~/astro_sky_image_vqa/LMM_outputs_n150/'),
    ('archive-light', '~/astro_sky_image_vqa/LMM_outputs_n150_archive_light/'),
]

# model subdirectory -> display name, in the order they should appear.  Same
# low tier as the other figure notebooks.
model_names = {
    'chatgpt_api':  'GPT',
    'gemini':       'Gemini',
    'claude_haiku': 'Claude',
}

# How many figures the released dataset is meant to hold once generation is
# finished.  The counts below are over whatever is actually on disk now; this
# is only used to say how far along that is.
target_n_figures = 3000

# Print the raw text of a few responses that could not be parsed, per model and
# run, so the count in (3) can be eyeballed rather than trusted.  0 to skip.
n_examples_to_show = 2

In [2]:
import os, json, glob, pickle, collections
import pandas as pd

# The response parser the rest of the paper's notebooks score with -- using it
# here rather than a local json.loads means "unparsable" in this notebook is
# exactly "unparsable" in the accuracy figures.
import sys as _sys
_sys.path.insert(0, os.path.abspath(os.path.join('..')))
from utils.parse_lmm_output_utils import (answer_for, expected_key_from_format,
                                          parse_llm_dual_json, PLACEHOLDER)

qa_jsons_dir = os.path.expanduser(qa_jsons_dir)


class _StubObject:
    """Stand-in for a class whose module is not installed in this kernel."""
    def __init__(self, *args, **kwargs):
        # enums and dataclasses are rebuilt by CALLING the class, so this has
        # to swallow whatever the pickle hands it rather than inherit
        # object.__init__, which refuses arguments.
        self._args = args
    def __setstate__(self, state):
        self.__dict__.update(state if isinstance(state, dict) else {})


class _TolerantUnpickler(pickle.Unpickler):
    """
    Load a run pickle without the vendor SDKs installed.

    Each pickle carries the raw provider response alongside the qa list, so a
    plain pickle.load needs `anthropic` and `google.genai` importable -- which
    would make this notebook runnable only in the env the runs were made in.
    Nothing here touches those objects: every field read below ('raw answer',
    'Response', 'format', ...) is a plain string put there by the run notebook.
    So unknown classes are swapped for an inert stub instead of raising.
    """
    def find_class(self, module, name):
        try:
            return super().find_class(module, name)
        except (ImportError, AttributeError):
            return type(name, (_StubObject,), {})


def load_qa(pickle_path):
    """The list of qa entries in a run pickle, whatever else it carries."""
    with open(pickle_path, 'rb') as f:
        payload = _TolerantUnpickler(f).load()
    return payload[0] if isinstance(payload, (list, tuple)) else payload


def is_unparsable(entry):
    """
    True when no answer could be recovered from this response.

    Mirrors the scoring notebooks exactly: the raw response, keyed by whatever
    the prompt's own format instruction asked for.
    """
    raw = entry.get('raw answer') or entry.get('Response')
    return answer_for(raw, expected_key_from_format(entry.get('format'))) is None


def unparsable_reason(entry):
    """Which of the two failure modes it was -- for the breakdown table."""
    raw = (entry.get('raw answer') or entry.get('Response') or '')
    if not raw.strip():
        return 'empty response'
    answer_str, _ = parse_llm_dual_json(raw)
    if not answer_str or answer_str == PLACEHOLDER:
        return 'no parsable json'
    return 'json, but no answer value'

## 1. Every question in the released dataset

Counted straight out of the qa jsons.  Each json is a json *string* holding the
figure's parameters plus a `VQA` block; the questions sit two levels down, as
`VQA[level][question type][question name]`, so the count is the number of leaf
entries.  Every figure in this dataset is single-panel, so there is no
per-panel multiplication to worry about -- the assert below is what says so.


In [3]:
qa_json_paths = sorted(glob.glob(os.path.join(qa_jsons_dir, '*_qa.json')))
print('qa jsons found:', len(qa_json_paths), 'in', qa_jsons_dir)

per_figure = {}                       # vqa_id -> question count
by_level   = collections.Counter()    # 'Level 1' -> question count
by_kind    = collections.Counter()    # ('Level 1', 'Figure-level questions') -> count
n_panels   = collections.Counter()

for path in qa_json_paths:
    with open(path) as f:
        d = json.loads(json.load(f))          # the file is a json-encoded string
    vqa_id = os.path.basename(path).removesuffix('_qa.json')
    n_panels[sum(1 for k in d if k.startswith('plot'))] += 1

    n = 0
    for level, kinds in d['VQA'].items():
        for kind, questions in kinds.items():
            n += len(questions)
            by_level[level] += len(questions)
            by_kind[(level, kind)] += len(questions)
    per_figure[vqa_id] = n

total_questions_dataset = sum(per_figure.values())
n_figures_dataset       = len(per_figure)

assert set(n_panels) == {1}, 'multi-panel figures present: %s' % dict(n_panels)

print()
print('figures         :', n_figures_dataset,
      '(of a target %d -- %.1f%%)' % (target_n_figures,
                                      100 * n_figures_dataset / target_n_figures))
print('TOTAL QUESTIONS :', total_questions_dataset)
print('per figure      : %.2f mean' % (total_questions_dataset / n_figures_dataset))

qa jsons found: 2001 in /Users/jnaiman/astro_sky_image_vqa/VQA_full/qa_jsons/

figures         : 2001 (of a target 3000 -- 66.7%)
TOTAL QUESTIONS : 47357
per figure      : 23.67 mean


In [4]:
# Why the per-figure count is not a single number: contour and sky panels get
# different question sets, so the total is not n_figures * a constant.
print('questions per figure, and how many figures have that many:')
for n, c in sorted(collections.Counter(per_figure.values()).items()):
    print('   %2d questions  x  %4d figures  =  %6d' % (n, c, n * c))
print()

print('where those questions come from:')
rows = [{'level': lvl, 'question type': kind, 'questions': n,
         'per figure': n / n_figures_dataset}
        for (lvl, kind), n in sorted(by_kind.items())]
df_breakdown = pd.DataFrame(rows)
display(df_breakdown.style.format({'per figure': '{:.2f}'}).hide(axis='index'))

questions per figure, and how many figures have that many:
   21 questions  x   667 figures  =   14007
   25 questions  x  1334 figures  =   33350

where those questions come from:


level,question type,questions,per figure
Level 1,Figure-level questions,20010,10.00
Level 1,Plot-level questions,12673,6.33
Level 2,Plot-level questions,10672,5.33
Level 3,Plot-level questions,4002,2.00


## 2. What was actually asked -- the n150 runs

The dataset above is the pool; the runs are a 150-figure sample of it, asked of
three models.  Both runs use the same 150 figures and the same questions, so
`original` and `archive-light` should come out identical -- if they do not, one
of the runs is incomplete.


In [5]:
run_rows   = []          # one per (run, model)
run_qa     = {}          # (run, model) -> list of (vqa_id, entry), kept for part 3

for run_label, run_path in runs:
    run_path = os.path.expanduser(run_path)
    for mdir, mname in model_names.items():
        pkls = sorted(glob.glob(os.path.join(run_path, mdir, '*_qa.pickle')))
        if not pkls:
            print('[WARN] no pickles in', os.path.join(run_path, mdir))
            continue

        entries = []
        for fp in pkls:
            vqa_id = os.path.basename(fp).removesuffix('_qa.pickle')
            for e in load_qa(fp):
                entries.append((vqa_id, e))
        run_qa[(run_label, mname)] = entries

        run_rows.append({'run': run_label, 'model': mname,
                         'figures': len(pkls), 'questions': len(entries)})

df_runs = pd.DataFrame(run_rows)
display(df_runs.pivot(index='model', columns='run', values='questions')
        .reindex(list(model_names.values())))

total_questions_asked = int(df_runs['questions'].sum())
print('TOTAL QUESTIONS ASKED (both n150 runs, all three models):',
      total_questions_asked)
for run_label, _ in runs:
    sub = df_runs[df_runs['run'] == run_label]
    print('   %-14s %6d questions over %d figures x %d models'
          % (run_label, sub['questions'].sum(), sub['figures'].max(), len(sub)))

run,archive-light,original
model,,
GPT,3550,3550
Gemini,3550,3550
Claude,3550,3550


TOTAL QUESTIONS ASKED (both n150 runs, all three models): 21300
   original        10650 questions over 150 figures x 3 models
   archive-light   10650 questions over 150 figures x 3 models


In [6]:
# Sanity check: the runs should be asking the dataset's own questions, so the
# per-figure count in a run must match the count in that figure's qa json.
mismatches = []
for (run_label, mname), entries in run_qa.items():
    per_fig_run = collections.Counter(vqa_id for vqa_id, _ in entries)
    for vqa_id, n in per_fig_run.items():
        if per_figure.get(vqa_id) != n:
            mismatches.append((run_label, mname, vqa_id, n, per_figure.get(vqa_id)))

if mismatches:
    print('[WARN] %d figure(s) asked a different number of questions than their'
          ' qa json holds:' % len(mismatches))
    for m in mismatches[:10]:
        print('   run=%s model=%s %s: asked %s, json has %s' % m)
else:
    print('OK: every figure in every run was asked exactly the question set in'
          ' its qa json.')

OK: every figure in every run was asked exactly the question set in its qa json.


## 3. Unparsable answers, per model and per run

The denominator is the *asked* count from part 2, not the dataset total.
`unparsable %` is what fraction of a model's responses in that run yielded no
answer at all -- the questions that are dropped from, rather than scored wrong
in, the accuracy figures.


In [7]:
bad_rows     = []
bad_entries  = collections.defaultdict(list)   # (run, model) -> unparsable entries
reason_rows  = []

for (run_label, mname), entries in run_qa.items():
    bad = [(vqa_id, e) for vqa_id, e in entries if is_unparsable(e)]
    bad_entries[(run_label, mname)] = bad

    bad_rows.append({'run': run_label, 'model': mname,
                     'questions': len(entries), 'unparsable': len(bad),
                     'unparsable %': 100 * len(bad) / len(entries)})

    for _, e in bad:
        reason_rows.append({'run': run_label, 'model': mname,
                            'reason': unparsable_reason(e)})

df_bad = (pd.DataFrame(bad_rows)
          .set_index(['model', 'run'])
          .reindex(pd.MultiIndex.from_product(
              [list(model_names.values()), [r[0] for r in runs]],
              names=['model', 'run']))
          .reset_index())

display(df_bad.style.format({'unparsable %': '{:.2f}%'}).hide(axis='index'))

print('TOTAL UNPARSABLE (both runs, all three models): %d of %d questions'
      ' (%.2f%%)' % (df_bad['unparsable'].sum(), df_bad['questions'].sum(),
                     100 * df_bad['unparsable'].sum() / df_bad['questions'].sum()))

model,run,questions,unparsable,unparsable %
GPT,original,3550,6,0.17%
GPT,archive-light,3550,12,0.34%
Gemini,original,3550,3,0.08%
Gemini,archive-light,3550,3,0.08%
Claude,original,3550,5,0.14%
Claude,archive-light,3550,1,0.03%


TOTAL UNPARSABLE (both runs, all three models): 30 of 21300 questions (0.14%)


In [8]:
# Which failure mode, and which questions.  Both are worth a line in the paper:
# a parse failure concentrated on one question is a prompt problem, one spread
# evenly is a model problem.
if reason_rows:
    df_reason = pd.DataFrame(reason_rows)
    print('failure mode:')
    display(pd.crosstab([df_reason['model'], df_reason['run']],
                        df_reason['reason']))

q_rows = [{'run': run_label, 'model': mname,
           'question': (e.get('question') or '')[:70]}
          for (run_label, mname), bad in bad_entries.items() for _, e in bad]
if q_rows:
    print('which questions they were (top 10):')
    display(pd.DataFrame(q_rows)['question'].value_counts().head(10)
            .rename('unparsable responses').to_frame())

failure mode:


reason                json, but no answer value  no parsable json
model  run                                                       
Claude archive-light                          0                 1
       original                               0                 5
GPT    archive-light                          8                 4
       original                               6                 0
Gemini archive-light                          0                 3
       original                               0                 3

which questions they were (top 10):


,unparsable responses
question,
What is the mean value of the data along the color-axis in this figure,9
What are the values for each of the tick marks on the y-axis?,5
What is the underlying distribution used to create the data in this fi,4
What is the y-axis title of the plot in this figure?,3
"What is the angular width of the sky region shown in this figure, meas",2
What are the values for each of the tick marks on the x-axis?,2
What is the median value of the data along the color-axis in this figu,1
What is the maximum right ascension value covered by the right ascensi,1
What is the maximum declination value covered by the declination axis,1


In [9]:
# The raw text of a few of them, so the count above can be checked by eye
# rather than taken on faith.
for (run_label, mname), bad in sorted(bad_entries.items()):
    if not bad or n_examples_to_show <= 0:
        continue
    print('=' * 78)
    print('%s / %s  -- %d unparsable' % (mname, run_label, len(bad)))
    print('=' * 78)
    for vqa_id, e in bad[:n_examples_to_show]:
        print('  %s  |  question: %s' % (vqa_id, e.get('question')))
        print('  asked for key : %r' % expected_key_from_format(e.get('format')))
        print('  raw response  : %r' % ((e.get('raw answer') or e.get('Response') or '')[:400]))
        print()

Claude / archive-light  -- 1 unparsable
  vqa_000104  |  question: What is the underlying distribution used to create the data in this figure?
  asked for key : 'distribution'
  raw response  : 'Looking at this astronomical survey image, I can observe:\n\n1. **Multiple bright point sources** (stars/objects) scattered across the field, with some particularly bright ones visible around coordinates (10h, 50\'30") and (10h, 51\')\n\n2. **Continuous granular background** that shows realistic astronomical noise and variations typical of actual sky observations\n\n3. **Spatial distribution patterns** th'

GPT / archive-light  -- 12 unparsable
  vqa_000014  |  question: What are the values for each of the tick marks on the x-axis?
  asked for key : 'xtick values'
  raw response  : '{"xtick values":[["$5^{\\\\mathrm{h}}07^{\\\\mathrm{m}}22^{\\\\mathrm{s}}$","$20^{\\\\mathrm{s}}$","$18^{\\\\mathrm{s}}$","$16^{\\\\mathrm{s}}$","$14^{\\\\mathrm{s}}$]]}\n\n{"explanation":"The x-axis tick labels vis

## The three numbers

In [10]:
print('(1) questions in the released dataset : %6d  (over %d figures)'
      % (total_questions_dataset, n_figures_dataset))
print('(2) questions asked in the n150 runs   : %6d  (%s x %d models)'
      % (total_questions_asked,
         ' + '.join(lab for lab, _ in runs), len(model_names)))
print('(3) unparsable responses               : %6d  (%.2f%% of (2))'
      % (df_bad['unparsable'].sum(),
         100 * df_bad['unparsable'].sum() / df_bad['questions'].sum()))
print()
for _, r in df_bad.iterrows():
    print('      %-7s %-14s %4d of %4d  (%.2f%%)'
          % (r['model'], r['run'], r['unparsable'], r['questions'],
             r['unparsable %']))

(1) questions in the released dataset :  47357  (over 2001 figures)
(2) questions asked in the n150 runs   :  21300  (original + archive-light x 3 models)
(3) unparsable responses               :     30  (0.14% of (2))

      GPT     original          6 of 3550  (0.17%)
      GPT     archive-light    12 of 3550  (0.34%)
      Gemini  original          3 of 3550  (0.08%)
      Gemini  archive-light     3 of 3550  (0.08%)
      Claude  original          5 of 3550  (0.14%)
      Claude  archive-light     1 of 3550  (0.03%)
